# LangChain: Agents

## Goal of this notebook

Large Language Models can generate text, but they cannot directly perform actions such as calling APIs, executing code, or retrieving external information.

To overcome this limitation we use **LLM agents**.

Agents combine a language model with external tools that the model can choose to use during reasoning.

In this notebook we explore how to build an agent in LangChain that can:

- reason about a task
- choose an appropriate tool
- execute the tool
- incorporate the result into the final answer

### Agent Architecture

The workflow of an LLM agent typically follows this loop:

User Question  
↓  
LLM Reasoning  
↓  
Tool Selection  
↓  
Tool Execution  
↓  
Observation  
↓  
Further Reasoning  
↓  
Final Answer

## Environment Setup

We start by importing the required libraries and configuring the OpenAI API key.

This notebook uses:

- LangChain
- OpenAI chat models
- tool abstractions
- agent frameworks

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Use a current model name to avoid migration warnings in modern LangChain.
llm_model = "gpt-4o-mini"

### Model Configuration

We specify the language model to use throughout this notebook.

The `temperature` parameter controls the randomness of model outputs:

- `0` produces deterministic, consistent responses
- higher values produce more creative but less predictable outputs

In [3]:
from contextlib import redirect_stdout
import io
import wikipedia

from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.globals import set_debug
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


In [4]:
llm = ChatOpenAI(temperature=0, model=llm_model)

agent_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Use the available tools when they help you answer accurately.",
        ),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)


def build_agent(tools):
    agent_runnable = create_openai_tools_agent(llm, tools, agent_prompt)
    return AgentExecutor(
        agent=agent_runnable,
        tools=tools,
        verbose=True,
        handle_parsing_errors=True,
    )

## Defining Tools

Tools are functions that the agent can call to perform specific tasks.

Examples of tools include:

- calculators
- search engines
- APIs
- custom Python functions

Each tool has:

- a name
- a description
- a callable function

The language model uses the description to decide when the tool should be used.


## Creating the Agent

The agent combines:

- a language model
- a set of tools
- a reasoning strategy

LangChain agents typically follow the **ReAct framework**, where the model alternates between reasoning and acting.

The agent decides:

1. what action to take
2. which tool to use
3. when to stop and produce a final answer

## Creating Custom Tools

In this section we define custom tools that the agent can use.

A tool is essentially a Python function wrapped with metadata such as:

- a name
- a description
- an input schema

The description is particularly important because the LLM uses it to decide when the tool should be used during reasoning.


In this notebook we define two tools:

- `calculator` — evaluates a math expression and returns the result
- `wikipedia_search` — looks up a topic on Wikipedia and returns a short summary

In [5]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic Python math expression. Example input: 0.25 * 300."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


@tool
def wikipedia_search(query: str) -> str:
    """Look up a topic on Wikipedia and return a short summary."""
    try:
        return wikipedia.summary(query, sentences=2, auto_suggest=False)
    except Exception:
        matches = wikipedia.search(query, results=5)
        if matches:
            return "Possible Wikipedia matches: " + ", ".join(matches)
        return "No Wikipedia results found."


tools = [calculator, wikipedia_search]

### Initializing the Agent

We now pass the tools to `build_agent`, which creates an `AgentExecutor` backed by the language model.

The agent will have access to both tools and will decide which one to use based on the user's question.

In [6]:
agent = build_agent(tools)

## Agent Reasoning Process

Agents typically follow the **ReAct reasoning pattern**.

The reasoning loop looks like this:

Thought → Action → Observation → Thought

The model first reasons about the problem, then selects a tool to execute an action.

After observing the result of the tool, the model continues reasoning until it can produce the final answer.

## Running the Agent

We now run the agent on a user query.

When the agent receives a question it will:

1. analyze the question
2. determine whether a tool is needed
3. select the appropriate tool
4. execute the tool
5. incorporate the result into the final answer

In [7]:
agent.invoke({"input": "What is 25% of 300?"})



> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '0.25 * 300'}`


75.025% of 300 is 75.

> Finished chain.


{'input': 'What is 25% of 300?', 'output': '25% of 300 is 75.'}

## Agents with Multiple Tools

Agents become significantly more powerful when they have access to multiple tools.

Instead of relying only on the language model, the agent can dynamically choose the best tool for the task.

For example:

- a calculator tool for math problems
- a search tool for retrieving information
- custom APIs for domain-specific queries

The agent decides which tool to use based on the tool descriptions and the user's question.

In the example below we ask the agent a factual question that requires retrieving information from an external source.

The agent will automatically select the `wikipedia_search` tool to look up the answer.

In [8]:
question = "Tom M. Mitchell is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). What book did he write?"
result = agent.invoke({"input": question})



> Entering new AgentExecutor chain...

Invoking: `wikipedia_search` with `{'query': 'Tom M. Mitchell'}`


Tom Michael Mitchell (born August 9, 1951) is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). He is a founder and former chair of the Machine Learning Department at CMU. Mitchell is known for his contributions to the advancement of machine learning, artificial intelligence, and cognitive neuroscience and is the author of the textbook Machine Learning.Tom M. Mitchell is the author of the textbook "Machine Learning."

> Finished chain.


In [9]:
@tool
def python_repl(code: str) -> str:
    """Execute Python code for simple data processing. Use print(...) to show the final answer."""
    try:
        scope = globals().copy()
        scope["__builtins__"] = __builtins__
        buffer = io.StringIO()
        with redirect_stdout(buffer):
            exec(code, scope, scope)
        output = buffer.getvalue().strip()
        return output if output else "Code executed successfully with no printed output."
    except Exception as e:
        return f"Error: {e}"


agent = build_agent([python_repl])

In [10]:
customer_list = [["Harrison", "Chase"], 
                 ["Lang", "Chain"],
                 ["Dolly", "Too"],
                 ["Elle", "Elem"], 
                 ["Geoff","Fusion"], 
                 ["Trance","Former"],
                 ["Jen","Ayai"]
                ]

In [11]:
agent.invoke(
    {
        "input": f"""Sort these customers by last name and then first name.
Use Python and print the final sorted list: {customer_list}"""
    }
)



> Entering new AgentExecutor chain...

Invoking: `python_repl` with `{'code': "customers = [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]\n\n# Sort by last name (index 0) and then by first name (index 1)\nsorted_customers = sorted(customers, key=lambda x: (x[0], x[1]))\n\nprint(sorted_customers)"}`


[['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Harrison', 'Chase'], ['Jen', 'Ayai'], ['Lang', 'Chain'], ['Trance', 'Former']]The sorted list of customers by last name and then first name is:

```
[['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Harrison', 'Chase'], ['Jen', 'Ayai'], ['Lang', 'Chain'], ['Trance', 'Former']]
```

> Finished chain.


{'input': "Sort these customers by last name and then first name.\nUse Python and print the final sorted list: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]",
 'output': "The sorted list of customers by last name and then first name is:\n\n```\n[['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Harrison', 'Chase'], ['Jen', 'Ayai'], ['Lang', 'Chain'], ['Trance', 'Former']]\n```"}

### View Detailed Outputs of the Chains

We can enable debug mode to see the full reasoning trace of the agent.

This shows each step the agent takes, including:

- the thought process
- the tool selected
- the input passed to the tool
- the observation returned

In [12]:
set_debug(True)
agent.invoke(
    {
        "input": f"""Sort these customers by last name and then first name.
Use Python and print the final sorted list: {customer_list}"""
    }
)
set_debug(False)

[chain/start] [chain:AgentExecutor] Entering Chain run with input:
{
  "input": "Sort these customers by last name and then first name.\nUse Python and print the final sorted list: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]"
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad>] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad> > chain:RunnableParallel<agent_scratchpad>] Entering Chain run with input:
{
  "input": ""
}
[chain/start] [chain:AgentExecutor > chain:RunnableSequence > chain:RunnableAssign<agent_scratchpad> > chain:RunnableParallel<agent_scratchpad> > chain:RunnableLambda] Entering Chain run with input:
{
  "input": ""
}
[chai

## Define My Own Tool

Beyond the built-in tools, we can define any custom Python function as a tool.

In this example we create a `time` tool that returns today's date.

This demonstrates how straightforward it is to extend the agent with domain-specific capabilities.

In [13]:
from datetime import date

In [14]:
@tool
def time(text: str) -> str:
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())

### Adding the Tool to the Agent

We add the `time` tool to the existing set of tools and reinitialize the agent.

The agent now has access to three tools and will select the appropriate one depending on the question.

In [15]:
agent = build_agent(tools + [time])

In [16]:
try:
    result = agent.invoke({"input": "whats the date today?"})
except Exception:
    print("exception on external access")



> Entering new AgentExecutor chain...

Invoking: `time` with `{'text': ''}`


2026-03-16Today's date is March 16, 2026.

> Finished chain.


## Conclusion

In this notebook we explored how to build LLM agents using LangChain.

Key takeaways:

- Agents extend language models by enabling them to use external tools.
- Tools allow LLMs to perform actions such as calculations, API calls, and information retrieval.
- The ReAct reasoning framework allows agents to iteratively think, act, and observe results.
- Agents enable more complex workflows compared to simple prompt-based systems.

Agent architectures are widely used in modern AI systems, including autonomous assistants, research tools, and workflow automation systems.